# week06 · 노트북 02 — 톨루엔 50분자 상자: Packmol → 게이트 → data → in.toluene → 첫 NPT

덱 6-3 · 19장(P6-2) · 6-4. `data/Toluene.mol`(연구 플랫폼 단분자 좌표) + `data/Toluene_gaff2.json`(GAFF2 · AM1-BCC 아티팩트) → ρ0 0.2 g/cm³ · tolerance 3.0 Å 로 Packmol → 품질 게이트 → `toluene_box.data` → `in.toluene` → LAMMPS 최소화 + NPT.

실행: week06 폴더에서 `claude` → 「`@notebooks/02_packmol_box.ipynb` 를 실행하고 결과를 요약해」(Claude Code 가 week06 폴더에서 `python ../run_nb.py notebooks/02_packmol_box.ipynb` 로 실행) · 또는 JupyterLab 에서 직접. 순서: 1/4 환경 확인 → ▶ 2/4 상자 만들기(스크립트 호출) → ▶ 3/4 게이트 · data 확인 → ▶ 4/4 실행 · 밀도 곡선. 결과 = `out/toluene_box.data` · `in.toluene` · `build_report.json` · `log.toluene`.

In [ ]:
# 1/4 환경 확인 · 기본 = 로컬 mdai(lmp · packmol 포함) · Claude Code 실행 · 아래 if 는 (선택) Colab 에서만 저장소 받기 · LAMMPS 설치
import os, subprocess, sys, pathlib
if 'google.colab' in sys.modules:
    if not pathlib.Path('/content/mooc').exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/sentirain7/2026_MOOC_Lecture_Materials.git','/content/mooc'], check=False)
    os.chdir('/content/mooc/week06')
    # conda-forge 고정 버전 (덱 5장) — 약 3분
    subprocess.run('pip -q install condacolab && python -c "import condacolab; condacolab.install()"', shell=True)
    subprocess.run('conda install -y -q -c conda-forge "lammps=2025.07.22=cpu_py311*nompi*" packmol=21.2.1', shell=True)
else:
    os.chdir(pathlib.Path.cwd() if pathlib.Path('scripts').exists() else pathlib.Path(__file__).resolve().parents[1])
import shutil; LMP = shutil.which('lmp') or shutil.which('lmp_serial'); PACKMOL = shutil.which('packmol')
print('week06 폴더:', os.getcwd()); print('lmp:', LMP, '· packmol:', PACKMOL, '— None 이면 expected_output 의 로그로 진행')


In [ ]:
# ▶ 2/4 상자 만들기 — mol → pdb → Packmol → 게이트 → data → in (scripts/build_toluene_box.py · 덱 22 · 23 · 25장)
import subprocess, sys, json
args = [sys.executable, 'scripts/build_toluene_box.py', '--n', '50', '--rho0', '0.2', '--tol', '3.0']
if PACKMOL: args += ['--packmol', PACKMOL]
r = subprocess.run(args, capture_output=True, text=True); print(r.stdout[-1500:], r.stderr[-800:])
rep = json.load(open('out/build_report.json')); print(json.dumps(rep, indent=1, ensure_ascii=False))

In [ ]:
# ▶ 3/4 게이트 · data 확인 (덱 25장 검증 3)
pk, d = rep['packmol'], rep['data']
checks = {'게이트 pass (exit 0 · Success! · 거리 ≥ 0.9 tol)': pk['pass'], '원자 수 750 = 50 × 15': d['atoms'] == 750, '|Σq| < 1e-4': abs(d['charge_sum']) < 1e-4, 'L 33.69 Å': abs(rep['L_A'] - 33.69) < 0.05}
for k, v in checks.items(): print('✔' if v else '✘', k)
print('분자간 최소 거리', pk['min_intermolecular_distance_A'], 'Å · 기준', pk['threshold_A'], 'Å')
print(open('out/toluene_box.data').read().split('Masses')[0])   # 헤더 4항목 (덱 10장)

In [ ]:
# ▶ 4/4 LAMMPS 실행 — 최소화 + NPT 6 ps + NVT 3 ps (약 30 초 · 덱 18장) · 없으면 expected_output 로그
import subprocess, pathlib, re, shutil
run = pathlib.Path('out/run'); run.mkdir(exist_ok=True)
if LMP:
    shutil.copy('out/toluene_box.data', run / 'toluene_box.data')
    s = open('out/in.toluene').read().replace('run            20000', 'run            6000').replace('run            10000', 'run            3000')
    (run / 'in.toluene').write_text(s)
    r = subprocess.run([LMP, '-in', 'in.toluene', '-log', 'log.toluene'], cwd=run, capture_output=True, text=True); print('exit', r.returncode)
    LOG = run / 'log.toluene'
else:
    LOG = pathlib.Path('expected_output/log.toluene'); print('LAMMPS 없음 → expected_output/log.toluene')
rows, hdr, phase = [], None, 0
for l in LOG.read_text().splitlines():
    if l.strip().startswith('Step'): phase += 1; hdr = l.split(); continue
    if l.startswith('Loop time'): hdr = None; continue
    if hdr and re.match(r'^\s*\d+\s', l) and len(l.split()) == len(hdr): rows.append((phase, dict(zip(hdr, map(float, l.split())))))
npt = [r for p, r in rows if p == 2]
print(f"NPT: 밀도 {npt[0]['Density']:.4f} → {npt[-1]['Density']:.4f} g/cm³ · T {npt[-1]['Temp']:.1f} K · 아직 상승 중 → 프로토콜은 2 ns (덱 15장)")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
ax[0].plot([r['Step'] for p, r in rows if p >= 2], [r['Temp'] for p, r in rows if p >= 2], color='#D9534F'); ax[0].axhline(298, ls='--', color='gray'); ax[0].set_ylabel('T (K)')
ax[1].plot([r['Step'] for p, r in rows if p >= 2], [r['Density'] for p, r in rows if p >= 2], color='#4F7FB0'); ax[1].set_ylabel('ρ (g/cm³)'); ax[1].set_xlabel('step (1 fs)')
plt.tight_layout(); plt.savefig('out/toluene_run.png', dpi=120); plt.show()

## 확인
- `build_report.json` 의 `pass: true` · atoms 750 · |Σq| < 1e−4
- `log.toluene` 에 `Total wall time` · Density 가 0.2 보다 커졌는가(6 ps 에서 약 0.31)
- `WARNING: System is not charge neutral, net charge = -5e-05` 는 허용 범위(검증 2)
- 못 돌리면 `expected_output/` 의 data · in · log 로 W07